In [1]:
from dataclasses import dataclass
from openai import OpenAI
import os

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests accross all inference providers """

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS = [
    Provider("OpenAI", "OPENAI_API_KEY", True, None, "gpt-4o-mini"),
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider: Provider) -> OpenAI:

    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    
    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url,
    )


def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.


In [2]:
def llm_reply(prompt: str) -> str:
    provider = select_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": prompt,
        }]
    )

    return result.choices[0].message.content

In [4]:
prompt = "Who was the PM of UK before 2020 ? Answer in a single sentence"

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Error: {e}")




Using Groq provider
The Prime Minister of the United Kingdom before 2020 was Boris Johnson, who assumed office on 24 July 2019.


## HW: Modify the implementation of providers to add a OPEN Router provider as well. 

In [5]:
def chat_reply(messages: list[dict]) -> str:
    provider = select_provider()

    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=1000,
        messages=messages,
    )

    return result.choices[0].message.content

In [6]:
conversation = []

In [7]:
conversation.append({"role": "user", "content": "What is the capital of France?"})

In [9]:
reply_from_llm = chat_reply(conversation)
print(reply_from_llm)

conversation.append({
    "role": "assistant",
    "content": reply_from_llm,
})


The capital of France is **Paris**.


In [10]:
conversation

[{'role': 'user', 'content': 'What is the capital of France?'},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'}]

In [11]:
conversation.append({
    "role": "user",
    "content": "what is their GDP ? ",
})

In [12]:
conversation

[{'role': 'user', 'content': 'What is the capital of France?'},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'},
 {'role': 'user', 'content': 'what is their GDP ? '}]

In [13]:
llm_reply = chat_reply(conversation)
print(llm_reply)

**France’s Gross Domestic Product (GDP)** (latest figures available up to 2024)

| Metric | Value | Year | Notes |
|--------|-------|------|-------|
| **Nominal GDP** | **≈ $2.9 trillion USD** | 2023 (IMF estimate) | Ranks around 7th‑8th largest economy in the world. |
| **GDP (Purchasing‑Power‑Parity, PPP)** | **≈ $3.4 trillion international dollars** | 2023 (World Bank) | Adjusts for price‑level differences; places France 5th‑6th globally. |
| **GDP per capita (nominal)** | **≈ $43,000 USD** | 2023 (IMF) | Based on total population ~67 million. |
| **GDP growth rate** | **+0.6 %** (real growth) | 2023 (OECD) | Modest expansion after the pandemic‑era slowdown. |

### Sources
- **International Monetary Fund (IMF)** – World Economic Outlook Database, April 2024 edition.  
- **World Bank** – World Development Indicators, accessed 2024.  
- **Organisation for Economic Co‑operation and Development (OECD)** – Economic Outlook, 2024.

### Quick takeaway
France is a high‑income, advanced econ

## Key takeaways

- A bare **model** is a one-shot function: text in, text out, no memory.
- A **chatbot** = model + a **transcript list** of `{"role", "content"}` dicts. Sending that list back each turn *is* memory.
- A real call is still text-in / text-out, just wrapped: a messages list goes in, `choices[0].message.content` comes out (OpenAI dialect).
- **All three providers speak the OpenAI dialect** — Groq & OpenRouter just reuse the `openai` SDK with a different `base_url`, so one call path covers them all.